# 02 - Divisao treino / validacao / teste

Saida: `data/annotations/splits.csv` com colunas `id,paciente,caminho,classe,conjunto`.

In [6]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ANNOT_PATH   = PROJECT_ROOT / 'data' / 'annotations' / 'annotations.csv'
OUT_PATH     = PROJECT_ROOT / 'data' / 'annotations' / 'splits.csv'

RATIOS = {'treino': 0.70, 'validacao': 0.15, 'teste': 0.15}
SEED   = 42

## 1. Carrega imagens validas

In [7]:
df = pd.read_csv(ANNOT_PATH)
df = df[(df.status == 'ok') & df.spectrogram_path.notna()].copy()
df = df[df.patient_id != 'PN10']  # PN10 tem apenas 20 canais

print(f'Imagens validas: {len(df)}')
df.groupby(['patient_id', 'class']).size().unstack(fill_value=0)

Imagens validas: 137


class,ictal,normal,pos_ictal,pre_ictal
patient_id,,,,
PN00,4,5,4,4
PN03,2,2,2,2
PN05,3,3,3,3
PN06,5,5,5,5
PN07,1,1,1,1
PN09,3,3,3,3
PN11,1,1,1,1
PN12,4,4,4,4
PN13,3,3,3,3


## 2. Divisao por paciente

Pacientes ordenados por numero de imagens (maior primeiro). Cada paciente vai para o conjunto mais distante da sua meta.

In [8]:
counts = df.patient_id.value_counts()
total  = counts.sum()
filled = {k: 0 for k in RATIOS}
patient_set = {}

for pid, n in counts.items():
    target = max(RATIOS, key=lambda k: RATIOS[k] * total - filled[k])
    patient_set[pid] = target
    filled[target] += n

df['conjunto'] = df.patient_id.map(patient_set)

for k in RATIOS:
    pids = sorted(p for p, s in patient_set.items() if s == k)
    print(f'{k:10s} {filled[k]:4d} imagens ({filled[k]/total:5.1%}) | {pids}')

# Garantia: nenhum paciente em mais de um conjunto
assert df.groupby('patient_id').conjunto.nunique().max() == 1

treino       97 imagens (70.8%) | ['PN00', 'PN03', 'PN05', 'PN06', 'PN07', 'PN11', 'PN12', 'PN14']
validacao    20 imagens (14.6%) | ['PN09', 'PN16']
teste        20 imagens (14.6%) | ['PN13', 'PN17']


## 3. Oversampling apenas no treino

Cada classe minoritaria do treino e duplicada (amostragem com reposicao) ate o tamanho da classe majoritaria.

In [9]:
train = df[df.conjunto == 'treino']
other = df[df.conjunto != 'treino']

max_n = train['class'].value_counts().max()
parts = []
for cls, g in train.groupby('class'):
    extra = g.sample(max_n - len(g), replace=True, random_state=SEED) if len(g) < max_n else g.iloc[:0]
    parts += [g, extra]
train_bal = pd.concat(parts)

print('Treino antes :', train['class'].value_counts().to_dict())
print('Treino depois:', train_bal['class'].value_counts().to_dict())

Treino antes : {'normal': 25, 'ictal': 24, 'pre_ictal': 24, 'pos_ictal': 24}
Treino depois: {'ictal': 25, 'normal': 25, 'pos_ictal': 25, 'pre_ictal': 25}


## 4. Salva splits.csv

In [10]:
out = pd.concat([train_bal, other], ignore_index=True)
out = pd.DataFrame({
    'paciente': out.patient_id,
    'caminho' : out.spectrogram_path,
    'classe'  : out['class'].str.upper(),
    'conjunto': out.conjunto,
})
out.insert(0, 'id', range(1, len(out) + 1))
out.to_csv(OUT_PATH, index=False)

print(f'Salvo em {OUT_PATH} ({len(out)} linhas)')
display(out.groupby(['conjunto', 'classe']).size().unstack(fill_value=0))
out.head()

Salvo em /home/orlandomota/Documents/repositorios_github/epilepsy-detection/data/annotations/splits.csv (140 linhas)


classe,ICTAL,NORMAL,POS_ICTAL,PRE_ICTAL
conjunto,,,,
teste,5,5,5,5
treino,25,25,25,25
validacao,5,5,5,5


,id,paciente,caminho,classe,conjunto
0,1,PN00,data/processed/spectrograms/PN00/PN00-1_seizur...,ICTAL,treino
1,2,PN00,data/processed/spectrograms/PN00/PN00-2_seizur...,ICTAL,treino
2,3,PN00,data/processed/spectrograms/PN00/PN00-4_seizur...,ICTAL,treino
3,4,PN00,data/processed/spectrograms/PN00/PN00-5_seizur...,ICTAL,treino
4,5,PN03,data/processed/spectrograms/PN03/PN03-1_seizur...,ICTAL,treino


## 5. Testes de integridade e vazamento

In [11]:
# Testes de integridade e vazamento
s = pd.read_csv(OUT_PATH)
VALID = {'NORMAL', 'PRE_ICTAL', 'ICTAL', 'POS_ICTAL'}
checks = {}

# Integridade
checks['colunas corretas']          = list(s.columns) == ['id', 'paciente', 'caminho', 'classe', 'conjunto']
checks['ids unicos']                = s.id.is_unique
checks['sem valores nulos']         = not s.isna().any().any()
checks['classes validas']           = set(s.classe) <= VALID
checks['conjuntos validos']         = set(s.conjunto) <= set(RATIOS)
checks['todos os arquivos existem'] = all((PROJECT_ROOT / c).exists() for c in s.caminho.unique())
checks['todas as imagens usadas']   = set(s.caminho) == set(df.spectrogram_path)

# Vazamento
checks['paciente em 1 so conjunto'] = s.groupby('paciente').conjunto.nunique().max() == 1
checks['imagem em 1 so conjunto']   = s.groupby('caminho').conjunto.nunique().max() == 1

# Oversampling restrito ao treino
vt = s[s.conjunto != 'treino']
checks['val/teste sem duplicatas']  = vt.caminho.is_unique
checks['val/teste inalterados']     = len(vt) == (df.conjunto != 'treino').sum()
checks['treino balanceado']         = s[s.conjunto == 'treino'].classe.value_counts().nunique() == 1

for name, ok in checks.items():
    print(f"[{'OK' if ok else 'FALHA'}] {name}")
assert all(checks.values()), 'Ha testes com falha'
print('\nTodos os testes passaram.')

[OK] colunas corretas
[OK] ids unicos
[OK] sem valores nulos
[OK] classes validas
[OK] conjuntos validos
[OK] todos os arquivos existem
[OK] todas as imagens usadas
[OK] paciente em 1 so conjunto
[OK] imagem em 1 so conjunto
[OK] val/teste sem duplicatas
[OK] val/teste inalterados
[OK] treino balanceado

Todos os testes passaram.
